## 1. Initialize Project Environment
Import libraries for sequence fetching, **multiple sequence alignment**, and distance calculation.

> **Important**: Distances must be computed from **aligned** sequences to be biologically meaningful. Comparing unaligned sequences gives random noise because position N in one species is not homologous to position N in another.

In [1]:
from __future__ import annotations

import itertools
import logging
import time
import urllib.parse
import urllib.request
from dataclasses import dataclass, asdict
from io import StringIO
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
from Bio import Entrez, SeqIO, AlignIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio.Align import MultipleSeqAlignment

logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")

print("pandas", pd.__version__)
print("numpy", np.__version__)
try:
    import Bio

    print("biopython", Bio.__version__)
except Exception as exc:
    logging.error("Biopython import failed: %s", exc)

pandas 2.2.3
numpy 2.1.3
biopython 1.85


## 2. Define Configuration Parameters
Centralize sequence accessions, export paths, and distance calculation options.

In [2]:
@dataclass
class DistanceConfig:
    handle: str
    email: str
    export_dir: Path = Path("artifacts")
    trim_bp: Optional[int] = 1500  # Trim for Clustal Omega API (max ~2000 recommended)
    ebi_api_url: str = "https://www.ebi.ac.uk/Tools/services/rest/clustalo"
    poll_interval: int = 5
    accessions: List[str] = None

    def __post_init__(self):
        if self.accessions is None:
            # TP53 sequences from 10 different vertebrate species
            self.accessions = [
                "NM_000546.6",  # Homo sapiens (human)
                "NM_011640.3",  # Mus musculus (mouse)
                "NM_131327.2",  # Danio rerio (zebrafish)
                "NM_001003210.1",  # Canis lupus familiaris (dog)
                "NM_213824.3",  # Sus scrofa (pig)
                "NM_174201.2",  # Bos taurus (cattle)
                "NM_205264.1",  # Gallus gallus (chicken)
                "NM_030989.3",  # Rattus norvegicus (rat)
                "XM_016931470.3",  # Pan troglodytes (chimpanzee)
                "NM_001047151.2",  # Macaca mulatta (rhesus macaque)
            ]

    def describe(self) -> Dict[str, str]:
        info = asdict(self)
        info["export_dir"] = str(info["export_dir"])
        return info


CONFIG = DistanceConfig(
    handle="AndreiCod",
    email="student@example.com",
)
CONFIG.describe()

{'handle': 'AndreiCod',
 'email': 'student@example.com',
 'export_dir': 'artifacts',
 'trim_bp': 1500,
 'ebi_api_url': 'https://www.ebi.ac.uk/Tools/services/rest/clustalo',
 'poll_interval': 5,
 'accessions': ['NM_000546.6',
  'NM_011640.3',
  'NM_131327.2',
  'NM_001003210.1',
  'NM_213824.3',
  'NM_174201.2',
  'NM_205264.1',
  'NM_030989.3',
  'XM_016931470.3',
  'NM_001047151.2']}

In [3]:
def fetch_sequences(cfg: DistanceConfig) -> List[SeqRecord]:
    """Fetch sequences from NCBI Entrez."""
    Entrez.email = cfg.email
    records = []

    for accession in cfg.accessions:
        try:
            logging.info("Fetching %s...", accession)
            handle = Entrez.efetch(
                db="nucleotide", id=accession, rettype="fasta", retmode="text"
            )
            record = SeqIO.read(handle, "fasta")
            handle.close()
            records.append(record)
        except Exception as e:
            logging.warning("Failed to fetch %s: %s", accession, e)

    logging.info("Successfully fetched %d sequences", len(records))
    return records


# Fetch all sequences
sequences = fetch_sequences(CONFIG)
print(f"\nFetched {len(sequences)} sequences:")
for rec in sequences:
    print(f"  {rec.id}: {len(rec.seq)} bp")

[INFO] Fetching NM_000546.6...
[INFO] Fetching NM_011640.3...
[INFO] Fetching NM_131327.2...
[INFO] Fetching NM_001003210.1...
[INFO] Fetching NM_213824.3...
[INFO] Fetching NM_174201.2...
[INFO] Fetching NM_205264.1...
[INFO] Fetching NM_030989.3...
[INFO] Fetching XM_016931470.3...
[INFO] Fetching NM_001047151.2...
[INFO] Successfully fetched 10 sequences



Fetched 10 sequences:
  NM_000546.6: 2512 bp
  NM_011640.3: 1781 bp
  NM_131327.2: 2233 bp
  NM_001003210.1: 1174 bp
  NM_213824.3: 1849 bp
  NM_174201.2: 2175 bp
  NM_205264.1: 1555 bp
  NM_030989.3: 1792 bp
  XM_016931470.3: 2634 bp
  NM_001047151.2: 2184 bp


In [4]:
# Save raw sequences and prepare trimmed versions for MSA
DATA_DIR = Path(f"../../../data/work/{CONFIG.handle}/lab04")
DATA_DIR.mkdir(parents=True, exist_ok=True)
fasta_path = DATA_DIR / "tp53_multi_sequences.fasta"
SeqIO.write(sequences, fasta_path, "fasta")
logging.info("Saved %d sequences to %s", len(sequences), fasta_path)


# Trim sequences for Clustal Omega API (handles ~1500bp well)
def trim_sequences(records: List[SeqRecord], max_len: Optional[int]) -> List[SeqRecord]:
    """Trim sequences to max length for MSA API."""
    if not max_len:
        return records
    trimmed = []
    for rec in records:
        if len(rec.seq) > max_len:
            new_rec = rec[:max_len]
            new_rec.description = f"{rec.description} [trimmed to {max_len}bp]"
        else:
            new_rec = rec
        trimmed.append(new_rec)
    return trimmed


sequences_trimmed = trim_sequences(sequences, CONFIG.trim_bp)
print(f"\nTrimmed sequences for MSA (max {CONFIG.trim_bp} bp):")
for rec in sequences_trimmed:
    print(f"  {rec.id}: {len(rec.seq)} bp")

[INFO] Saved 10 sequences to ../../../data/work/AndreiCod/lab04/tp53_multi_sequences.fasta



Trimmed sequences for MSA (max 1500 bp):
  NM_000546.6: 1500 bp
  NM_011640.3: 1500 bp
  NM_131327.2: 1500 bp
  NM_001003210.1: 1174 bp
  NM_213824.3: 1500 bp
  NM_174201.2: 1500 bp
  NM_205264.1: 1500 bp
  NM_030989.3: 1500 bp
  XM_016931470.3: 1500 bp
  NM_001047151.2: 1500 bp


## 3. Run Multiple Sequence Alignment (Clustal Omega)
**Critical step**: Align sequences before computing distances. Without alignment, we compare non-homologous positions!

In [5]:
def submit_clustalo_job(sequences: List[SeqRecord], cfg: DistanceConfig) -> str:
    """Submit sequences to EBI Clustal Omega REST API."""
    fasta_io = StringIO()
    SeqIO.write(sequences, fasta_io, "fasta")
    fasta_string = fasta_io.getvalue()

    params = {
        "email": cfg.email,
        "sequence": fasta_string,
        "outfmt": "clustal_num",
    }

    data = urllib.parse.urlencode(params).encode("utf-8")
    url = f"{cfg.ebi_api_url}/run"

    logging.info("Submitting %d sequences to Clustal Omega API...", len(sequences))
    req = urllib.request.Request(url, data=data, method="POST")
    with urllib.request.urlopen(req, timeout=60) as response:
        job_id = response.read().decode("utf-8").strip()

    logging.info("Job submitted: %s", job_id)
    return job_id


def poll_job_status(job_id: str, cfg: DistanceConfig, max_wait: int = 300) -> str:
    """Poll EBI API until job completes."""
    url = f"{cfg.ebi_api_url}/status/{job_id}"
    elapsed = 0

    while elapsed < max_wait:
        with urllib.request.urlopen(url, timeout=30) as response:
            status = response.read().decode("utf-8").strip()

        if status == "FINISHED":
            logging.info("Job %s finished successfully", job_id)
            return status
        elif status in ("FAILURE", "ERROR"):
            raise RuntimeError(f"Clustal Omega job failed: {status}")

        logging.info("Job status: %s (waiting %ds...)", status, cfg.poll_interval)
        time.sleep(cfg.poll_interval)
        elapsed += cfg.poll_interval

    raise TimeoutError(f"Job {job_id} did not complete within {max_wait}s")


def fetch_alignment_result(job_id: str, cfg: DistanceConfig) -> str:
    """Fetch alignment result from completed job."""
    url = f"{cfg.ebi_api_url}/result/{job_id}/aln-clustal_num"
    with urllib.request.urlopen(url, timeout=60) as response:
        result = response.read().decode("utf-8")
    return result


# Run MSA
EXPORT_DIR = CONFIG.export_dir
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

job_id = submit_clustalo_job(sequences_trimmed, CONFIG)
poll_job_status(job_id, CONFIG)
alignment_text = fetch_alignment_result(job_id, CONFIG)

# Save MSA result
msa_path = EXPORT_DIR / "task1_msa_result.clustal"
with open(msa_path, "w") as f:
    f.write(alignment_text)

print(f"[OK] MSA saved to: {msa_path.resolve()}")
print(f"\nFirst 500 characters:\n{alignment_text[:500]}")

[INFO] Submitting 10 sequences to Clustal Omega API...
[INFO] Job submitted: clustalo-R20251230-142014-0509-10688328-p1m
[INFO] Job clustalo-R20251230-142014-0509-10688328-p1m finished successfully


[OK] MSA saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/04_phylogenetics/assignments/artifacts/task1_msa_result.clustal

First 500 characters:
CLUSTAL O(1.2.4) multiple sequence alignment


NM_000546.6         ------CTCAAAAGTCT---------------------------------AGAGCCACCG	21
NM_011640.3         TTTCCCCTCCCACGTGCTCACCCTGGCTAAAGTTCTGTAGCTTCAGTTCA-TTGGGACCA	59
NM_131327.2         ------------------------------------------------------------	0
NM_001003210.1      ------------------------------------------------------------	0
NM_213824.3         ---------AAAAGTCC---------------------------------AGGGCCACCA	18
NM_174201.2         ---------TAAAGT


In [6]:
# Load and verify alignment
alignment = AlignIO.read(msa_path, "clustal")
logging.info(
    "Loaded MSA: %d sequences × %d positions",
    len(alignment),
    alignment.get_alignment_length(),
)

print(
    f"Alignment dimensions: {len(alignment)} seqs × {alignment.get_alignment_length()} bp"
)
print(f"\nSequence IDs in alignment:")
for rec in alignment:
    print(f"  {rec.id}")

[INFO] Loaded MSA: 10 sequences × 1859 positions


Alignment dimensions: 10 seqs × 1859 bp

Sequence IDs in alignment:
  NM_000546.6
  NM_011640.3
  NM_131327.2
  NM_001003210.1
  NM_213824.3
  NM_174201.2
  NM_205264.1
  NM_030989.3
  XM_016931470.3
  NM_001047151.2


## 4. Compute Distances from Aligned Sequences
Now compute p-distance from the **aligned** sequences (homologous positions).

In [7]:
def compute_aligned_pairwise_distances(alignment: MultipleSeqAlignment) -> pd.DataFrame:
    """Compute pairwise distances from aligned sequences (p-distance excluding gaps)."""
    rows = []
    records = list(alignment)

    for rec_i, rec_j in itertools.combinations(records, 2):
        seq_i = str(rec_i.seq)
        seq_j = str(rec_j.seq)

        # Count differences at non-gap positions
        differences = 0
        compared = 0
        for a, b in zip(seq_i, seq_j):
            if a != "-" and b != "-":  # Only compare non-gap positions
                compared += 1
                if a != b:
                    differences += 1

        p_dist = differences / compared if compared > 0 else np.nan

        rows.append(
            {
                "seq_a": rec_i.id,
                "seq_b": rec_j.id,
                "compared_positions": compared,
                "differences": differences,
                "p_distance": p_dist,
            }
        )

    return pd.DataFrame(rows)


pairwise_df = compute_aligned_pairwise_distances(alignment)
print(f"Computed {len(pairwise_df)} pairwise distances from aligned sequences")
pairwise_df

Computed 45 pairwise distances from aligned sequences


,seq_a,seq_b,compared_positions,differences,p_distance
0,NM_000546.6,NM_011640.3,1423,370,0.260014
1,NM_000546.6,NM_131327.2,1316,642,0.487842
2,NM_000546.6,NM_001003210.1,1168,193,0.165240
3,NM_000546.6,NM_213824.3,1447,272,0.187975
4,NM_000546.6,NM_174201.2,1469,297,0.202178
5,NM_000546.6,NM_205264.1,1254,551,0.439394
6,NM_000546.6,NM_030989.3,1436,362,0.252089
7,NM_000546.6,XM_016931470.3,1381,3,0.002172
8,NM_000546.6,NM_001047151.2,1437,71,0.049408
9,NM_011640.3,NM_131327.2,1299,647,0.498075


## 5. Create Distance Matrix and Export
Format as symmetric matrix and validate biological expectations.

In [8]:
def create_distance_matrix(df: pd.DataFrame, value_col: str) -> pd.DataFrame:
    """Create symmetric distance matrix from pairwise dataframe."""
    ids = sorted(set(df["seq_a"]) | set(df["seq_b"]))
    matrix = pd.DataFrame(0.0, index=ids, columns=ids)

    for _, row in df.iterrows():
        matrix.loc[row["seq_a"], row["seq_b"]] = row[value_col]
        matrix.loc[row["seq_b"], row["seq_a"]] = row[value_col]

    return matrix


p_distance_matrix = create_distance_matrix(pairwise_df, "p_distance")

# Species mapping for display
SPECIES = {
    "NM_000546.6": "Human",
    "NM_011640.3": "Mouse",
    "NM_131327.2": "Zebrafish",
    "NM_001003210.1": "Dog",
    "NM_213824.3": "Pig",
    "NM_174201.2": "Cattle",
    "NM_205264.1": "Chicken",
    "NM_030989.3": "Rat",
    "XM_016931470.3": "Chimpanzee",
    "NM_001047151.2": "Rhesus",
}

# Display with species names
display_matrix = p_distance_matrix.copy()
display_matrix.index = [SPECIES.get(x, x) for x in display_matrix.index]
display_matrix.columns = [SPECIES.get(x, x) for x in display_matrix.columns]

print("=== DISTANCE MATRIX (from MSA-aligned sequences) ===\n")
display_matrix.round(3)

=== DISTANCE MATRIX (from MSA-aligned sequences) ===



,Human,Dog,Rhesus,Mouse,Rat,Zebrafish,Cattle,Chicken,Pig,Chimpanzee
Human,0.000,0.165,0.049,0.260,0.252,0.488,0.202,0.439,0.188,0.002
Dog,0.165,0.000,0.171,0.226,0.235,0.466,0.190,0.400,0.173,0.167
Rhesus,0.049,0.171,0.000,0.270,0.257,0.494,0.211,0.463,0.205,0.054
Mouse,0.260,0.226,0.270,0.000,0.118,0.498,0.297,0.456,0.271,0.235
Rat,0.252,0.235,0.257,0.118,0.000,0.488,0.300,0.457,0.279,0.223
Zebrafish,0.488,0.466,0.494,0.498,0.488,0.000,0.490,0.495,0.477,0.492
Cattle,0.202,0.190,0.211,0.297,0.300,0.490,0.000,0.435,0.155,0.184
Chicken,0.439,0.400,0.463,0.456,0.457,0.495,0.435,0.000,0.436,0.429
Pig,0.188,0.173,0.205,0.271,0.279,0.477,0.155,0.436,0.000,0.177
Chimpanzee,0.002,0.167,0.054,0.235,0.223,0.492,0.184,0.429,0.177,0.000


In [9]:
# Key pairwise distances
dm = display_matrix
key_distances = {
    "Human-Chimpanzee": dm.loc["Human", "Chimpanzee"],
    "Human-Rhesus": dm.loc["Human", "Rhesus"],
    "Mouse-Rat": dm.loc["Mouse", "Rat"],
    "Pig-Cattle": dm.loc["Pig", "Cattle"],
    "Chicken-Human": dm.loc["Chicken", "Human"],
    "Zebrafish-Human": dm.loc["Zebrafish", "Human"],
}

print("Key distances:")
for pair, dist in key_distances.items():
    print(f"  {pair}: {dist:.3f}")

Key distances:
  Human-Chimpanzee: 0.002
  Human-Rhesus: 0.049
  Mouse-Rat: 0.118
  Pig-Cattle: 0.155
  Chicken-Human: 0.439
  Zebrafish-Human: 0.488


## 6. Export Results
Save distance matrix and pairwise data to artifacts folder.

In [10]:
# Save pairwise distances
pairwise_path = EXPORT_DIR / "task1_pairwise_distances.csv"
pairwise_df.to_csv(pairwise_path, index=False)
print(f"[OK] Pairwise distances saved to: {pairwise_path.resolve()}")

# Save distance matrix
matrix_path = EXPORT_DIR / "task1_distance_matrix.csv"
p_distance_matrix.to_csv(matrix_path)
print(f"[OK] Distance matrix saved to: {matrix_path.resolve()}")

print(f"\nArtifacts saved to {EXPORT_DIR.resolve()}")

[OK] Pairwise distances saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/04_phylogenetics/assignments/artifacts/task1_pairwise_distances.csv
[OK] Distance matrix saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/04_phylogenetics/assignments/artifacts/task1_distance_matrix.csv

Artifacts saved to /home/rbals/git/daha-bdhb/BDHB-lab/labs/04_phylogenetics/assignments/artifacts
